# 2×2 Network-on-Chip — Traffic Benchmark

One Vivado simulation runs six independent workloads: LOW / MEDIUM / HIGH × UNIFORM / HOTSPOT. This notebook computes **end-to-end packet latency, throughput, backpressure and traffic matrices** and saves high-resolution PNG and vector PDF plots.

**Protocol:** latency starts at the *source accepted header flit* and ends at the *destination accepted tail flit* (3-flit packet). The golden model and FPGA area/timing analysis remain separate deliverables.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image

# Works when Jupyter is launched from repo root or analysis/.
HERE = Path.cwd()
ANALYSIS = HERE / "analysis" if (HERE / "analysis" / "analyze_noc.py").exists() else HERE
sys.path.insert(0, str(ANALYSIS))
from analyze_noc import *

DATA_DIR = ANALYSIS / "data"
OUT_DIR = ANALYSIS / "output"
FIG_DIR = OUT_DIR / "figures"
style()

def preview(name):
    display(Image(filename=str(FIG_DIR / (name + ".png"))))

print("Data folder:", DATA_DIR.resolve())

## 1. Load & validate data

In [ ]:
packets, cycles, summary = load_data(DATA_DIR)
metrics = compute_metrics(packets, cycles, summary)
write_summary(metrics, OUT_DIR)
display(metrics.round(4))

## 2. Mean packet latency vs accepted injection

Each plot is written as **PNG (300 dpi) and PDF (vector)** to `analysis/output/figures/`.

In [ ]:
plot_latency_vs_injection_rate(metrics, FIG_DIR)
preview("01_mean_latency_vs_injection_rate")

## 3. P95 packet latency vs accepted injection

Each plot is written as **PNG (300 dpi) and PDF (vector)** to `analysis/output/figures/`.

In [ ]:
plot_latency_vs_injection_rate(metrics, FIG_DIR, "P95LatencyCycles", "02_p95_latency_vs_injection_rate")
preview("02_p95_latency_vs_injection_rate")

## 4. Accepted input and completed output throughput vs threshold

Each plot is written as **PNG (300 dpi) and PDF (vector)** to `analysis/output/figures/`.

In [ ]:
plot_throughput(metrics, FIG_DIR)
preview("03_throughput_vs_generator_threshold")

## 5. Endpoint backpressure vs injection rate

Each plot is written as **PNG (300 dpi) and PDF (vector)** to `analysis/output/figures/`.

In [ ]:
plot_stalls(metrics, FIG_DIR)
preview("04_backpressure_vs_injection_rate")

## 6. Latency distributions for every configuration

Each plot is written as **PNG (300 dpi) and PDF (vector)** to `analysis/output/figures/`.

In [ ]:
plot_latency_box(packets, FIG_DIR)
preview("05_latency_distribution_boxplots")

## 7. High-rate latency ECDF

Each plot is written as **PNG (300 dpi) and PDF (vector)** to `analysis/output/figures/`.

In [ ]:
plot_high_rate_ecdf(packets, FIG_DIR)
preview("06_high_rate_latency_ecdf")

## 8. High-hotspot flit transfer activity

Each plot is written as **PNG (300 dpi) and PDF (vector)** to `analysis/output/figures/`.

In [ ]:
plot_flit_activity(cycles, FIG_DIR)
preview("07_high_hotspot_flit_activity")

## 9. In-flight and buffered packet count

Each plot is written as **PNG (300 dpi) and PDF (vector)** to `analysis/output/figures/`.

In [ ]:
plot_outstanding(cycles, FIG_DIR)
preview("08_high_hotspot_outstanding_packets")

## 10. Source-destination matrix: high-rate uniform

Each plot is written as **PNG (300 dpi) and PDF (vector)** to `analysis/output/figures/`.

In [ ]:
plot_matrix(packets, FIG_DIR, "uniform")
preview("09_traffic_matrix_high_uniform")

## 11. Source-destination matrix: high-rate hotspot

Each plot is written as **PNG (300 dpi) and PDF (vector)** to `analysis/output/figures/`.

In [ ]:
plot_matrix(packets, FIG_DIR, "hotspot")
preview("09_traffic_matrix_high_hotspot")

## 12. High-rate delivery by source

Each plot is written as **PNG (300 dpi) and PDF (vector)** to `analysis/output/figures/`.

In [ ]:
plot_source_fairness(packets, FIG_DIR)
preview("10_high_rate_deliveries_by_source")

## 13. Latency vs Manhattan hop count

Each plot is written as **PNG (300 dpi) and PDF (vector)** to `analysis/output/figures/`.

In [ ]:
plot_hop_latency(packets, FIG_DIR)
preview("11_latency_vs_hop_count")

## 14. High-hotspot cumulative accepted vs delivered

Each plot is written as **PNG (300 dpi) and PDF (vector)** to `analysis/output/figures/`.

In [ ]:
plot_packet_progress(cycles, FIG_DIR)
preview("12_high_hotspot_packet_progress")

## 15. Latency vs configured generator threshold

Each plot is written as **PNG (300 dpi) and PDF (vector)** to `analysis/output/figures/`.

In [ ]:
plot_grouped_rate_latency(metrics, FIG_DIR)
preview("13_latency_vs_generator_threshold")

## 16. Compare `FAST=0` vs `FAST=1` (optional)

Baseline and FAST require **two separate XSIM runs** with the same testbench, identical rate/phase settings and the same traffic-seed policy. This cell combines the logs if you have both sets.

In [ ]:
# Uncomment after saving the second run to analysis/data_fast/.
# packets_fast, cycles_fast, summary_fast = load_data(ANALYSIS / "data_fast")
# all_packets = pd.concat([packets, packets_fast], ignore_index=True)
# all_cycles = pd.concat([cycles, cycles_fast], ignore_index=True)
# all_summary = pd.concat([summary, summary_fast], ignore_index=True)
# all_metrics = compute_metrics(all_packets, all_cycles, all_summary)
# plot_latency_vs_injection_rate(all_metrics, FIG_DIR, name="baseline_vs_FAST_latency")
# preview("baseline_vs_FAST_latency")

### Interpretation cautions

- The source generator's hexadecimal threshold is **not** an offered packet rate. Use measured accepted packet injection on the primary latency-vs-injection plot.
- Warmup packets can complete in the measurement window. Latency is computed only from measurement-**injected** packets, including those delivered during drain.
- Endpoint stalls are not the same as internal router credit starvation. Full congestion attribution needs router-internal counters.
- A zero-loss test does not formally prove fairness or deadlock freedom.
- To prove paper-vs-baseline area/Fmax you must independently synthesize and implement both RTL variants.